# Ramulator 2.1 Tutorial: Simulating DDR5-4800

This notebook walks through Ramulator 2.1 with one DRAM configuration, **DDR5-4800AN** (`DDR5_16Gb_x8`):

1. **Configure** a DRAM device, a memory controller, and a memory system.
2. **Run** a first simulation with a simple out-of-order core.
3. **Read the statistics** the Python way and compare configurations.
4. **Explore** why latency-throughput curves bend backward under heavy write traffic.
5. **See** those read and write batches on the data bus in the RamViz trace visualizer.

**Before you start**
- Build Ramulator from the repository root: `./build.sh`.
- Run this notebook with the Python environment that has the `ramulator` package (for example `ramulator2-venv`).
- Section 5 also needs Node.js 20+ and a built visualizer (see that section).

The whole notebook runs in about two minutes on one CPU core.

## 0. Setup

In [ ]:
import contextlib
import csv
import os
import random
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt


def find_repo_root():
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "python" / "ramulator").is_dir() and (path / "examples").is_dir():
            return path
    raise RuntimeError("Start Jupyter from inside the Ramulator repository")


REPO = find_repo_root()
if str(REPO / "python") not in sys.path:
    sys.path.insert(0, str(REPO / "python"))

import ramulator

OUT = REPO / "examples" / "tutorial_output"  # files this notebook writes (git-ignored)
OUT.mkdir(exist_ok=True)

ORG_PRESET = "DDR5_16Gb_x8"
TIMING_PRESET = "DDR5_4800AN"


@contextlib.contextmanager
def quiet():
    # Ramulator's C++ components log to stderr; hide those lines while simulating.
    sys.stderr.flush()
    saved, devnull = os.dup(2), os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(saved)
        os.close(devnull)


print(f"Repository: {REPO}\nOutputs:    {OUT}")

## 1. Basic configuration

A Ramulator simulation has two top-level parts:

- a **frontend** that generates memory requests (a core model, a trace player, or a synthetic generator), and
- a **memory system** with one **memory controller** per channel. Each controller owns a **DRAM device** model and is built from a scheduler, a refresh manager, a row policy, and an address mapper.

```
Frontend ──requests──▶ Memory system ──▶ Controller (scheduler, refresh, row policy, address mapper) ──commands──▶ DRAM device
```

Everything is configured with Python objects. Let's start with the DRAM device. A device is a pair of presets: an
**organization** (density, width, banks, rows, ...) and a **timing** speed bin.

In [ ]:
print("Organization presets:", list(ramulator.dram.DDR5.org_presets))
print("DDR5-4800 timing presets:", [p for p in ramulator.dram.DDR5.timing_presets if "4800" in p])

`resolve()` returns the final organization and timing values, after the speed bin and any derived timings are applied.
Timings are in clock cycles (nCK); `tCK_ps` is the clock period.

In [ ]:
dram = ramulator.dram.DDR5(org_preset=ORG_PRESET, timing_preset=TIMING_PRESET)
org, timing = dram.resolve()
tCK_ns = timing["tCK_ps"] / 1000

print(f"Organization: {org['rank']} rank, {org['bankgroup']} bank groups x {org['bank']} banks, "
      f"{org['row']:,} rows x {org['column']:,} columns, x{org['dq']} devices on a {org['channel_width']}-bit channel")
print(f"Data rate {timing['rate']} MT/s, tCK = {tCK_ns:.3f} ns\n")

TIMINGS = {
    "nCL": "read latency (RD to data)",
    "nCWL": "write latency (WR to data)",
    "nBL": "burst length on the bus",
    "nRCD": "ACT to RD/WR",
    "nRP": "PRE to ACT",
    "nRAS": "ACT to PRE",
    "nCCDS": "RD to RD, different bank group",
    "nCCDL": "RD to RD, same bank group",
    "nRTW": "RD to WR (read-to-write turnaround)",
    "nWTRS": "write-to-read recovery, different bank group",
    "nWTRL": "write-to-read recovery, same bank group",
    "nREFI": "refresh interval",
    "nRFC": "refresh duration",
}
for name, meaning in TIMINGS.items():
    print(f"{name:6s} {timing[name]:6d} nCK = {timing[name] * tCK_ns:8.2f} ns   {meaning}")

Any organization field or timing can be **overridden** by name. Ramulator checks the names when it resolves the
configuration (with `resolve()`, or when a simulation starts): a typo is an error, not a silently ignored setting.

In [ ]:
dram_2r = ramulator.dram.DDR5(org_preset=ORG_PRESET, timing_preset=TIMING_PRESET, rank=2, nRCD=40)
org_2r, timing_2r = dram_2r.resolve()
print(f"rank: {org['rank']} -> {org_2r['rank']},  nRCD: {timing['nRCD']} -> {timing_2r['nRCD']}")

try:
    ramulator.dram.DDR5(org_preset=ORG_PRESET, timing_preset=TIMING_PRESET, nRCDD=40).resolve()
except ValueError as error:
    print("Rejected:", error)

## 2. First simulation

We drive the memory system with `SimpleO3`, a simple out-of-order core with a last-level cache. It replays an
instruction trace with one line per memory instruction:

```
<number of non-memory instructions before it> <load address> [<store address>]
```

The cell below writes a synthetic trace: most loads walk through four sequential streams, the rest go to random
addresses, and some instructions also store.

In [ ]:
def make_trace(path, *, lines=20_000, stream_share=0.7, store_share=0.2, seed=2026):
    rng = random.Random(seed)
    num_lines = 2**31 // 64  # 2 GB of 64-byte cache lines
    streams = [rng.randrange(num_lines) for _ in range(4)]
    with open(path, "w") as f:
        for _ in range(lines):
            if rng.random() < stream_share:
                i = rng.randrange(len(streams))
                streams[i] += 1
                load = streams[i]
            else:
                load = rng.randrange(num_lines)
            store = f" {rng.randrange(num_lines) * 64}" if rng.random() < store_share else ""
            f.write(f"{rng.randrange(8)} {load * 64}{store}\n")
    return path


TRACE = make_trace(OUT / "mixed.trace")
print(TRACE.read_text().splitlines()[:5])

Now build the system. The two `clock_ratio`s set the relative clock speeds: the frontend ticks 5 times for every
3 memory-system ticks, i.e. a 4 GHz core next to DDR5-4800's 2.4 GHz command clock.

In [ ]:
frontend = ramulator.frontend.SimpleO3(
    clock_ratio=5,
    traces=[str(TRACE)],
    num_expected_insts=200_000,  # stop after this many instructions retire
    translation=ramulator.translation.NoTranslation(max_addr=2**31),
)
controller = ramulator.controller.GenericDDR(
    dram=ramulator.dram.DDR5(org_preset=ORG_PRESET, timing_preset=TIMING_PRESET),
    scheduler=ramulator.scheduler.FRFCFS(),          # first-ready, first-come-first-served
    refresh_manager=ramulator.refresh_manager.AllBank(),
    row_policy=ramulator.row_policy.Open(),          # keep rows open after an access
    addr_mapper=ramulator.addr_mapper.RoBaRaCoCh(),  # address bits: row | bank | rank | column | channel
)
memory = ramulator.memory_system.GenericDRAM(
    clock_ratio=3,
    controllers=[controller],  # one controller per channel
    channel_mapper=ramulator.channel_mapper.CacheLineInterleave(),
)

with quiet():
    sim = ramulator.Simulation(frontend, memory)
    sim.run()

stats = sim.stats
ctrl = stats["memory_system"]["controller"]
print(f"Simulated {ctrl['cycles']:,} memory cycles ({ctrl['cycles'] * tCK_ns / 1000:.1f} us)")
print(f"Reads: {ctrl['num_read_reqs']:,}   Writes: {ctrl['num_write_reqs']:,}")
print(f"Average read latency: {ctrl['avg_read_latency']:.1f} cycles = {ctrl['avg_read_latency'] * tCK_ns:.1f} ns")

## 3. Getting statistics the Python way

`sim.stats` is a plain nested dictionary. The top level has `frontend` and `memory_system`; the controller's
counters sit under `stats["memory_system"]["controller"]`. (With several channels, `controller` becomes a list
with one dict per channel.)

In [ ]:
print("Top level:     ", list(stats))
print("Frontend:      ", sorted(stats["frontend"]))
print("Memory system: ", [k for k in stats["memory_system"] if k != "controller"])
print("Controller:    ", sorted(k for k, v in ctrl.items() if not isinstance(v, dict)))

Flattening the tree into dotted keys makes it easy to search:

In [ ]:
def flatten(tree, prefix=""):
    flat = {}
    for key, value in tree.items():
        if isinstance(value, dict):
            flat.update(flatten(value, f"{prefix}{key}."))
        else:
            flat[f"{prefix}{key}"] = value
    return flat


flat = flatten(stats)
for key, value in flat.items():
    if "latency" in key or "throughput" in key:
        print(f"{key:55s} {value}")

Raw counters become useful metrics with a few lines of Python. Keep such helpers next to your experiments:

In [ ]:
def summarize(stats):
    c, f = stats["memory_system"]["controller"], stats["frontend"]
    accesses = c["row_hits"] + c["row_misses"] + c["row_conflicts"]
    return {
        "row hit rate": c["row_hits"] / accesses,
        "row conflict rate": c["row_conflicts"] / accesses,
        "read latency (ns)": c["avg_read_latency"] * tCK_ns,
        "DRAM throughput (GB/s)": c["total_throughput_MBps"] / 1000,
        "core IPC": f["num_expected_insts"] / f["cycles_recorded_core_0"],
    }


for name, value in summarize(stats).items():
    print(f"{name:24s} {value:8.3f}")

(OUT / "first_run_stats.yaml").write_text(sim.stats_yaml)  # the same stats as YAML, for saving to disk
print("\nSaved", OUT / "first_run_stats.yaml")

Because configuration and results are both Python objects, a parameter sweep is just a loop. Here we compare two
row policies and three address mappings on the same trace and collect one dict per run:

In [ ]:
def make_simulation(trace, *, row_policy, addr_mapper, controller_plugins=()):
    controller = ramulator.controller.GenericDDR(
        dram=ramulator.dram.DDR5(org_preset=ORG_PRESET, timing_preset=TIMING_PRESET),
        scheduler=ramulator.scheduler.FRFCFS(),
        refresh_manager=ramulator.refresh_manager.AllBank(),
        row_policy=row_policy,
        addr_mapper=addr_mapper,
        controller_plugins=list(controller_plugins),
    )
    memory = ramulator.memory_system.GenericDRAM(
        clock_ratio=3, controllers=[controller], channel_mapper=ramulator.channel_mapper.CacheLineInterleave()
    )
    frontend = ramulator.frontend.SimpleO3(
        clock_ratio=5, traces=[str(trace)], num_expected_insts=200_000,
        translation=ramulator.translation.NoTranslation(max_addr=2**31),
    )
    return ramulator.Simulation(frontend, memory)


POLICIES = ("Open", "ClosedCAP")
MAPPERS = ("RoBaRaCoCh", "ChRaBaRoCo", "MOP4CLXOR")
results = []
for policy in POLICIES:
    for mapper in MAPPERS:
        with quiet():
            sim = make_simulation(
                TRACE,
                row_policy=getattr(ramulator.row_policy, policy)(),
                addr_mapper=getattr(ramulator.addr_mapper, mapper)(),
            )
            sim.run()
        results.append({"row policy": policy, "address mapper": mapper, **summarize(sim.stats)})

print(f"{'row policy':10s} {'mapper':11s} {'hit rate':>8s} {'conflicts':>9s} "
      f"{'latency':>9s} {'throughput':>11s} {'IPC':>6s}")
for r in results:
    print(f"{r['row policy']:10s} {r['address mapper']:11s} {r['row hit rate']:8.3f} {r['row conflict rate']:9.3f} "
          f"{r['read latency (ns)']:6.1f} ns {r['DRAM throughput (GB/s)']:6.2f} GB/s {r['core IPC']:6.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
width = 0.38
for ax, metric in zip(axes, ("DRAM throughput (GB/s)", "read latency (ns)", "row hit rate")):
    for j, policy in enumerate(POLICIES):
        values = [next(r[metric] for r in results if r["row policy"] == policy and r["address mapper"] == mapper)
                  for mapper in MAPPERS]
        ax.bar([i + (j - 0.5) * width for i in range(len(MAPPERS))], values, width, label=policy)
    ax.set_xticks(range(len(MAPPERS)), MAPPERS)
    ax.set_title(metric)
axes[0].legend(title="row policy")
plt.tight_layout()
plt.show()

**Try it:** which mapping keeps the sequential streams in one row, and which spreads them? Change `stream_share` in
`make_trace` and see how the ranking changes.

## 4. Why do latency-throughput curves bend backward?

A **latency-throughput curve** shows how the access latency grows as a memory system is loaded. Ramulator's
`LatencyThroughputTrace` frontend generates two kinds of traffic at once:

- **Streaming requests** create load. `nop_counter` sets the number of frontend cycles between them: a smaller
  value means more load. `read_ratio` sets the percentage of reads among them.
- **Probe requests** measure latency: random reads, each sent only after the previous one completes.

Each `nop_counter` value gives one point: (achieved throughput, average probe latency). The helper below runs one
point; it uses a pass-through address mapper because the generator already produces DRAM addresses.

In [ ]:
PROBES = 3000  # probe requests per point; more is smoother but slower
NOPS = (1000, 300, 100, 65, 50, 44, 41, 38, 36, 33, 20, 1)  # light load -> overload


def dram_layout(dram):
    # Address-generation parameters for LatencyThroughputTrace, read from the DRAM organization.
    levels = list(type(dram).levels)
    org, _ = dram.resolve()
    counts = [org.get(level.lower(), 1) for level in levels]
    row, col = levels.index("Row"), levels.index("Column")
    bank_positions = list(range(1, row))  # every level between Channel and Row
    if "BankGroup" in levels:  # cycle bank groups fastest, so streams use tCCD_S
        bank_positions.remove(levels.index("BankGroup"))
        bank_positions.append(levels.index("BankGroup"))
    bank_counts = [counts[p] for p in bank_positions]
    prefetch = type(dram).internal_prefetch_size
    total = 1
    for c in bank_counts:
        total *= c
    return dict(addr_vec_size=len(levels), bank_positions=bank_positions, bank_counts=bank_counts,
                total_bank_units=total, row_pos=row, col_pos=col, num_rows=counts[row],
                num_cols=counts[col], internal_prefetch_size=prefetch, num_cls=counts[col] // prefetch)


def direction_batches(trace_csv):
    # Average number of consecutive reads, and of consecutive writes, between bus direction switches.
    runs = {"R": [], "W": []}
    last, length = None, 0
    with open(trace_csv) as f:
        for row in csv.DictReader(f):
            if row["command"] in ("RD", "RDA", "WR", "WRA"):
                direction = row["command"][0]
                if direction == last:
                    length += 1
                else:
                    if last:
                        runs[last].append(length)
                    last, length = direction, 1
    return {d: sum(r) / len(r) if r else 0.0 for d, r in runs.items()}


def run_latp(read_ratio, nop, *, dram_overrides=None, controller_kwargs=None, controller_plugins=(),
             probes=PROBES, measure_batches=False):
    plugins = list(controller_plugins)
    if measure_batches:  # record every issued command to a CSV file (one file per channel)
        plugins.append(ramulator.controller_plugin.CmdTraceRecorder(path=str(OUT / "commands")))
    dram = ramulator.dram.DDR5(org_preset=ORG_PRESET, timing_preset=TIMING_PRESET, **(dram_overrides or {}))
    frontend = ramulator.frontend.LatencyThroughputTrace(
        clock_ratio=8, nop_counter=nop, read_ratio=read_ratio,
        num_probe_requests=probes, latency_sample_count=probes,
        stream_cls=64, stagger_stream_rows=True, **dram_layout(dram),
    )
    controller = ramulator.controller.GenericDDR(
        dram=dram,
        scheduler=ramulator.scheduler.FRFCFSRowHit(),  # prefers row hits
        refresh_manager=ramulator.refresh_manager.NoRefresh(),
        row_policy=ramulator.row_policy.Open(),
        addr_mapper=ramulator.addr_mapper.PassThroughAddrMapper(),
        controller_plugins=plugins,
        **(controller_kwargs or {}),
    )
    memory = ramulator.memory_system.GenericDRAM(
        clock_ratio=1, controllers=[controller], channel_mapper=ramulator.channel_mapper.PassThroughChannelMapper()
    )
    with quiet():
        sim = ramulator.Simulation(frontend, memory)
        sim.run()
        stats = sim.stats
        sim.finalize()  # flushes plugin outputs such as trace files
    tck_ns = dram.resolve()[1]["tCK_ps"] / 1000
    point = {
        "read_ratio": read_ratio,
        "nop": nop,
        "throughput": stats["memory_system"]["controller"]["total_throughput_MBps"] / 1000,
        "latency": stats["frontend"]["avg_probe_latency"] * tck_ns,
    }
    if measure_batches:
        trace = OUT / "commands.ch0"
        batches = direction_batches(trace)
        trace.unlink()  # the CSV is large; keep only the averages
        point.update(read_batch=batches["R"], write_batch=batches["W"])
    return point


def sweep(read_ratio, **kwargs):
    return [run_latp(read_ratio, nop, **kwargs) for nop in NOPS]


def backward_bend(curve):
    # Throughput lost between the curve's peak and its heaviest-load point, in percent.
    peak = max(p["throughput"] for p in curve)
    return 100 * (curve[-1]["throughput"] - peak) / peak


PEAK_GBPS = timing["rate"] * org["channel_width"] / 8 / 1000  # 4800 MT/s x 32 bits
print(f"Theoretical peak: {PEAK_GBPS:.1f} GB/s")

Sweep four read ratios, from read-only to half writes:

In [ ]:
def plot_curves(curves, title, colors=None, xlim=None):
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, (label, curve) in enumerate(curves.items()):
        color = colors[i] if colors else None
        ax.plot([p["throughput"] for p in curve], [p["latency"] for p in curve], "o-", label=label, color=color)
    ax.axvline(PEAK_GBPS, color="#cc7722", linestyle="--", label="theoretical peak")
    ax.set_xlabel("DRAM throughput (GB/s)")
    ax.set_ylabel("average probe latency (ns)")
    ax.set_xlim(*(xlim or (0, None)))
    ax.set_ylim(bottom=0)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()


t0 = time.time()
by_ratio = {f"{rr}% reads": sweep(rr) for rr in (100, 90, 70, 50)}
print(f"{len(by_ratio) * len(NOPS)} simulations in {time.time() - t0:.0f} s")
for label, curve in by_ratio.items():
    print(f"{label:10s} peak {max(p['throughput'] for p in curve):5.2f} GB/s, "
          f"backward bend at the heaviest load {backward_bend(curve):+5.1f}%")
plot_curves(by_ratio, "DDR5-4800AN", colors=["#b2182b", "#d6604d", "#4393c3", "#2166ac"])

The read-only curve behaves as expected: latency stays flat, then rises steeply near the peak. With more writes, the
curves saturate earlier, and the **50%-read curve bends backward**: past its peak, adding load *lowers* the
throughput.

### Hypothesis: the read-write turnaround

The data bus is shared by reads and writes, and switching its direction costs time:

- **read to write:** a WR must wait `nRTW` after a RD.
- **write to read:** a RD must wait `nCWL + nBL + nWTRS` after a WR to another bank group, or `nCWL + nBL + nWTRL`
  within the same bank group. The write data only reaches the bus `nCWL` after the WR and occupies it for `nBL`;
  the write-to-read recovery `nWTR` starts after that.

In [ ]:
print(f"RD -> WR:                 {timing['nRTW']:3d} nCK = {timing['nRTW'] * tCK_ns:5.2f} ns")
for name in ("nWTRS", "nWTRL"):
    cycles = timing["nCWL"] + timing["nBL"] + timing[name]
    print(f"WR -> RD (nCWL+nBL+{name}): {cycles:3d} nCK = {cycles * tCK_ns:5.2f} ns")

### Experiment 1: change the turnaround

If the turnaround causes the bend, changing it should change the bend. Each entry of `TURNAROUND` is a set of timing
overrides for the DDR5 device. Values below the JEDEC minimums are not physical, but they make a useful experiment:

- **no recovery** removes the extra read-to-write gap and the write-to-read recovery, but keeps `nCWL + nBL`;
- **near-free** also shrinks the write latency `nCWL`, so a switch in either direction costs about one burst.

In [ ]:
nBL = timing["nBL"]
TURNAROUND = {
    "default": {},
    "no recovery": {"nRTW": nBL, "nWTRS": 1, "nWTRL": 1},
    "near-free": {"nRTW": nBL, "nWTRS": 1, "nWTRL": 1, "nCWL": 1},
    "doubled": {"nRTW": 2 * timing["nRTW"], "nWTRS": 2 * timing["nWTRS"], "nWTRL": 2 * timing["nWTRL"]},
}

by_turnaround = {}
for label, overrides in TURNAROUND.items():
    t = {**timing, **overrides}
    by_turnaround[label] = sweep(50, dram_overrides=overrides)
    curve = by_turnaround[label]
    print(f"{label:12s} RD->WR {t['nRTW']:3d} nCK, WR->RD {t['nCWL'] + t['nBL'] + t['nWTRS']:3d}-"
          f"{t['nCWL'] + t['nBL'] + t['nWTRL']:3d} nCK:  peak {max(p['throughput'] for p in curve):5.2f} GB/s, "
          f"backward bend {backward_bend(curve):+5.1f}%")
plot_curves(by_turnaround, "DDR5-4800AN, 50% reads, different turnarounds (zoomed in)", xlim=(12, PEAK_GBPS + 0.5))

The bend grows with the turnaround, survives *no recovery*, and only disappears when a switch is nearly free. So the
turnaround sets the **price** of each direction switch. But it cannot explain why the throughput *drops* as load
increases: the price is the same at every load. Something must make the controller switch **more often** past the
peak.

### Experiment 2: count the direction switches

`run_latp(..., measure_batches=True)` attaches the `CmdTraceRecorder` controller plugin, which writes every issued
command to a CSV file, and measures how many reads (or writes) the controller issues in a row before the bus turns
around. Compare the default timing with the near-free turnaround:

In [ ]:
batch_runs = {label: sweep(50, dram_overrides=TURNAROUND[label], measure_batches=True)
              for label in ("default", "near-free")}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
x = range(len(NOPS))
for label, curve in batch_runs.items():
    ax1.plot(x, [p["throughput"] for p in curve], "o-", label=label)
    ax2.plot(x, [p["write_batch"] for p in curve], "o-", label=f"{label}: writes in a row")
    ax2.plot(x, [p["read_batch"] for p in curve], "s--", alpha=0.6, label=f"{label}: reads in a row")
default_curve = batch_runs["default"]
peak_index = max(range(len(NOPS)), key=lambda i: default_curve[i]["throughput"])
ax1.set_ylabel("DRAM throughput (GB/s)")
ax2.set_ylabel("commands per batch")
for ax in (ax1, ax2):
    ax.axvline(peak_index, color="gray", linestyle=":")
    ax.set_xticks(list(x), [str(n) for n in NOPS])
    ax.set_xlabel("nop_counter (light load -> overload); dotted: default peak")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("DDR5-4800AN, 50% reads")
plt.tight_layout()
plt.show()

Up to the peak, the controller issues longer and longer batches of reads and of writes, so each turnaround is shared
by more commands. Past the peak the batches get **shorter**, and they shrink the same way with the near-free
turnaround. Shorter batches mean more direction switches per byte: with the default turnaround every switch costs
time and the throughput drops; with a near-free turnaround the extra switches cost almost nothing and the curve stays
flat.

### Experiment 3: why do the batches shrink?

`GenericDDR` buffers writes and drains them in batches. It switches to writes when the write queue is fuller than
`wr_high_watermark` (or when there is no read to serve) and switches back once the write queue is below
`wr_low_watermark` **and reads are waiting**. Both are fractions of `write_buffer_size` (defaults: 0.8, 0.2 and 32).

- Below the peak, reads are not always waiting, so a drain can continue past the low watermark: long batches.
- At overload the read queue is always full, so every drain stops at the low watermark. A write batch is then about
  `(wr_high_watermark - wr_low_watermark) x write_buffer_size` = 0.6 x 32 ≈ 19 writes.

If this is right, a larger write queue should give longer batches at overload and a smaller bend:

In [ ]:
WR_HIGH, WR_LOW = 0.8, 0.2  # GenericDDR defaults

by_queue = {}
for size in (32, 64, 128):
    kwargs = {"write_buffer_size": size}
    curve = sweep(50, controller_kwargs=kwargs)
    overload = run_latp(50, NOPS[-1], controller_kwargs=kwargs, measure_batches=True)
    by_queue[f"write queue {size}"] = curve
    print(f"write queue {size:3d}: predicted batch {(WR_HIGH - WR_LOW) * size:5.1f}, "
          f"measured at overload {overload['write_batch']:5.1f} writes;  backward bend {backward_bend(curve):+5.1f}%, "
          f"latency at overload {curve[-1]['latency']:5.1f} ns")
plot_curves(by_queue, "DDR5-4800AN, 50% reads, write queue size (zoomed in)", xlim=(12, PEAK_GBPS + 0.5))

### Putting it together

The backward bend needs two ingredients:

1. **Shorter batches past the peak.** At overload the read queue is always full, so each write drain stops at the low
   watermark and the batch size drops to the watermark band times the write-queue size. The bus switches direction
   more often.
2. **A cost per switch.** Every switch pays the read-write turnaround, including the write latency `nCWL`.

Remove either one, with a near-free turnaround or with a large enough write queue, and the curve no longer bends.

### Exercises

1. **Watermarks.** Predict the batch size at overload for `wr_high_watermark=0.95, wr_low_watermark=0.05`, then check
   it with `run_latp(50, 1, controller_kwargs={...}, measure_batches=True)`. How large is the bend?
2. **Very large queues.** Use `write_buffer_size=512` and `read_buffer_size=512`. The bend disappears; what happens to
   the latency at overload, and why?
3. **More writes.** Sweep 30% reads. Is the bend stronger?
4. **Read queue.** Does `read_buffer_size` change the batch size at overload? Explain with the watermark rule.

In [ ]:
# Your experiments here, e.g.:
# point = run_latp(50, 1, controller_kwargs={"wr_high_watermark": 0.95, "wr_low_watermark": 0.05},
#                  measure_batches=True)
# print(point)

## 5. Seeing the read-write batches in the RamViz visualizer

Ramulator ships a browser-based trace visualizer, RamViz. It shows the issued commands on a timeline, the requests
they serve, the throughput over time, and the reconstructed state of every bank. Here we use it to *see* the
mechanism from section 4: batches of reads and writes on the data bus, separated by turnarounds.

**One-time setup** (from the repository root, needs Node.js 20 or newer):

```bash
cd visualizer
npm install
npm run build
```

Record two traces of the 50%-read workload at overload, one with the default 32-entry write queue and one with a
128-entry write queue. Each run writes a binary trace for RamViz (`BinTraceRecorder`) and a CSV command trace
(`CmdTraceRecorder`) that we use for a preview.

In [ ]:
VIZ_RUNS = {"write queue 32": 32, "write queue 128": 128}
VIZ = {}
for label, size in VIZ_RUNS.items():
    base = OUT / f"latp50_overload_wq{size}"
    point = run_latp(
        50, NOPS[-1], probes=500, controller_kwargs={"write_buffer_size": size},
        controller_plugins=[ramulator.controller_plugin.BinTraceRecorder(path=str(base)),
                            ramulator.controller_plugin.CmdTraceRecorder(path=str(base))],
    )
    with open(f"{base}.ch0") as f:
        bursts = [(int(row["clock"]), row["command"][0]) for row in csv.DictReader(f)
                  if row["command"] in ("RD", "RDA", "WR", "WRA")]
    VIZ[label] = {"ram2bin": Path(f"{base}.ch0.ram2bin"), "bursts": bursts}
    print(f"{label:16s} {point['throughput']:5.2f} GB/s  ->  {VIZ[label]['ram2bin']}")

**Preview.** This is what you are about to find in RamViz: 1.5 µs from the middle of each trace, every read (blue) and
write (teal) burst on the data bus, in RamViz's colors. Each run of one color is a batch; each white gap is a
turnaround.

In [ ]:
WINDOW_NS = 1500
fig, axes = plt.subplots(len(VIZ), 1, figsize=(14, 1.8 * len(VIZ)), sharex=True)
for ax, (label, run) in zip(axes, VIZ.items()):
    bursts = run["bursts"]
    t0 = bursts[len(bursts) // 2][0]
    t1 = t0 + round(WINDOW_NS / tCK_ns)
    window = [(clk, d) for clk, d in bursts if t0 <= clk < t1]
    switches = sum(a[1] != b[1] for a, b in zip(window, window[1:]))
    busy = len(window) * timing["nBL"] / (t1 - t0)
    for direction, color in (("R", "#1f5fb4"), ("W", "#0f9488")):
        ax.broken_barh([((clk - t0) * tCK_ns, timing["nBL"] * tCK_ns) for clk, d in window if d == direction],
                       (0, 1), color=color)
    ax.set_yticks([])
    ax.set_title(f"{label}: {switches} read/write switches, data bus busy {100 * busy:.0f}%   "
                 f"(RamViz window {t0 * tCK_ns:,.0f} - {t1 * tCK_ns:,.0f} ns = clk {t0:,} - {t1:,})",
                 loc="left", fontsize=10)
axes[-1].set_xlabel("time within the window (ns)")
plt.tight_layout()
plt.show()

Start the visualizer (`python -m ramulator visualize` does the same from a terminal). If one is already running on
the port, the cell reuses it.

In [ ]:
VIZ_PORT = 3000


def visualizer_running(port):
    try:
        with urllib.request.urlopen(f"http://localhost:{port}/api/health", timeout=1):
            return True
    except OSError:
        return False


def start_visualizer(port=VIZ_PORT):
    if visualizer_running(port):
        print(f"A visualizer is already running: http://localhost:{port}")
        return None
    if shutil.which("node") is None or not (REPO / "visualizer" / ".output" / "server" / "index.mjs").exists():
        print("Install Node.js 20+ and build the visualizer first (see the setup above).")
        return None
    env = {**os.environ, "PYTHONPATH": os.pathsep.join([str(REPO / "python"), os.environ.get("PYTHONPATH", "")])}
    process = subprocess.Popen([sys.executable, "-m", "ramulator", "visualize", "--port", str(port)],
                               cwd=REPO, env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if visualizer_running(port):
            print(f"Visualizer started: http://localhost:{port}")
            return process
        time.sleep(0.5)
    process.terminate()
    print("The visualizer did not start; run `python -m ramulator visualize` in a terminal to see why.")
    return None


viz = start_visualizer()
for label, run in VIZ.items():
    print(f"{label:16s} {run['ram2bin']}")

### Guided tour

1. **Open the first trace.** In the browser, drop `latp50_overload_wq32.ch0.ram2bin` on *Drop your trace here* (or
   click it to browse), press **Load trace**, then **Continue**. The trace opens with two panels: **Bank state** on
   top and **Commands** below. The inspector on the right shows the *Visible window*.
2. **Zoom out to about 1.5 µs.** The trace opens on its first 500 clk. Scroll the mouse wheel down over the timeline
   (about 14 notches) until *Visible window* reads about 3,500 clk. The pattern is the same throughout the trace;
   to see exactly the preview's window, drag the timeline (or the handle of the overview strip at the bottom) to the
   time printed above the preview.
3. **Find the batches.** In the **Commands** panel, look at the **DATA BUS** row under *Channel 0*: blue blocks are
   reads, teal blocks are writes. Each block is one batch; each white gap between blocks is a bus turnaround. Count
   the blocks in the window, and compare the gap after a write batch with the gap after a read batch.
4. **Read the utilization.** The **Bank state** panel's title line reports *data bus … utilized* for the visible
   window, and *open-but-starved* shows how much bank time is lost waiting for the bus.
5. **Compare.** Open `latp50_overload_wq128.ch0.ram2bin` in a second browser tab and repeat steps 1 to 4 at the same
   zoom.

**Questions**

- Which trace has longer batches? Is the batch length close to `(wr_high_watermark - wr_low_watermark) x
  write_buffer_size` from section 4?
- Why is the gap after a write batch wider than the gap after a read batch? (Hint: `nRTW` against
  `nCWL + nBL + nWTR`.)
- How much data-bus utilization does each trace reach, and how does that match the backward bend in section 4?

When you are done, stop the visualizer if this notebook started it:

In [ ]:
if viz is not None:
    viz.terminate()
    viz = None
    print("Visualizer stopped.")

## Where to go next

- `README.md`: the full user guide, including every frontend, controller plugin, and DRAM standard.
- Swap DDR5 for another standard, for example `ramulator.dram.LPDDR5(...)` with `ramulator.controller.LPDDR5(...)`.
- Write your own controller plugin to collect the statistic you need (README, section 8.2).